In [ ]:
import json
import google.generativeai as genai
from time import sleep
import re
import random

# === Cấu hình API Key Gemini ===
genai.configure(api_key="")

output_file = "/kaggle/working/clarification_data_enhanced.jsonl"
log_file = "/kaggle/working/clarification_enhanced_log.txt"
batch_size = 6
target_samples = 3000
temperature = 0.9
max_retry = 3

# === DANH SÁCH MODEL FALLBACK ===
model_priority_list = [
    'gemini-2.5-flash-lite',
    'gemini-2.5-flash', 
    'gemini-2.5-pro',
    'gemini-2.0-flash',
    'gemini-2.0-flash-lite',
]

max_output_tokens = 8192

class ClarificationDataGenerator:
    def __init__(self):
        self.vague_references = {
            "pronouns": ["ông ấy", "bà ấy", "vị đó", "người đó", "nơi đó", "sự kiện đó", "thời đó"],
            "time_periods": ["thời đó", "lúc đó", "hồi đó", "thời kỳ đó", "giai đoạn đó"],
            "locations": ["nơi đó", "chỗ đó", "vùng đó", "địa phương đó"],
            "events": ["sự kiện đó", "việc đó", "chuyện đó", "trận đó"],
            "people": ["vị vua đó", "nhân vật đó", "vị tướng đó", "ông vua đó"]
        }
        
        self.clarification_responses = [
            "Bạn có thể làm rõ câu hỏi được không? Ví dụ: bạn muốn hỏi về {specific_example}?",
            "Tôi cần thêm thông tin để trả lời chính xác. Bạn có thể nói rõ hơn về {missing_info} không?",
            "Câu hỏi của bạn khá chung chung. Bạn muốn tôi tập trung vào {aspect} cụ thể nào?",
            "Để tôi có thể hỗ trợ tốt nhất, bạn hãy cung cấp thêm chi tiết về {topic} nhé.",
            "Tôi cần biết thêm ngữ cảnh để trả lời chính xác. Bạn có thể mô tả rõ hơn về {context} không?",
            "Bạn có thể cho tôi biết cụ thể hơn về {detail} được không?",
            "Tôi chưa hiểu rõ bạn muốn hỏi về điều gì. Bạn có thể nói rõ hơn về {clarification_point}?",
            "Thông tin bạn cung cấp chưa đủ để tôi trả lời. Bạn có thể bổ sung thêm về {additional_info}?"
        ]
        
        self.specific_examples = {
            "vua": "vua Lý Thái Tổ, vua Quang Trung, hay vua Gia Long",
            "trận đánh": "trận Bạch Đằng, trận Chi Lăng, hay trận Điện Biên Phủ", 
            "triều đại": "nhà Lý, nhà Trần, hay nhà Nguyễn",
            "sự kiện": "khởi nghĩa Lam Sơn, phong trào Cần Vương, hay chiến tranh Đông Dương",
            "nhân vật": "Lê Lợi, Nguyễn Trãi, Quang Trung, hay Hồ Chí Minh",
            "thời kỳ": "thời Bắc thuộc, thời phong kiến, hay thời hiện đại",
            "địa điểm": "sông Bạch Đằng, thành Đa Bang, hay cố đô Huế"
        }

    def get_diverse_prompt(self, batch_size):
        prompt = f"""
HÃY TẠO {batch_size} MẪU HỘI THOẠI JSON CHO CHATBOT LỊCH SỬ VIỆT NAM VỚI CÂU HỎI MƠ HỒ CẦN LÀM RÕ:

YÊU CẦU ĐA DẠNG CAO:
1. MỖI câu hỏi phải mơ hồ theo cách KHÁC NHAU:
   - Sử dụng đại từ không rõ: "ông ấy", "bà ấy", "nơi đó"
   - Thiếu thông tin cơ bản: không có tên, năm, địa điểm
   - Câu hỏi quá rộng: "kể về", "nói về" mà không chỉ định
   - Ngữ cảnh không đầy đủ
   - Từ ngữ đa nghĩa

2. Đa dạng LOẠI MƠ HỒ:
   - Mơ hồ về NHÂN VẬT: "vị vua đó", "nhân vật lịch sử đó"
   - Mơ hồ về SỰ KIỆN: "trận đánh đó", "cuộc khởi nghĩa đó" 
   - Mơ hồ về THỜI GIAN: "thời đó", "lúc đó"
   - Mơ hồ về ĐỊA ĐIỂM: "nơi đó", "vùng đó"
   - Mơ hồ về NỘI DUNG: "kết quả đó", "ảnh hưởng đó"

3. Đa dạng CÂU TRẢ LỜI YÊU CẦU LÀM RÕ:
   - Sử dụng nhiều cách diễn đạt khác nhau
   - Đưa ra các ví dụ cụ thể khác nhau
   - Hỏi về các thông tin cụ thể khác nhau

VÍ DỤ ĐA DẠNG:
- "Kể về vị vua đó" → "Bạn có thể làm rõ... ví dụ vua Lý Thái Tổ, Quang Trung..."
- "Chiến thắng đó có ý nghĩa gì?" → "Tôi cần biết cụ thể... trận Bạch Đằng, Chi Lăng..."
- "Thời đó có gì đặc biệt?" → "Bạn muốn hỏi về thời kỳ nào... Bắc thuộc, phong kiến..."

ĐẢM BẢO: KHÔNG câu hỏi nào giống nhau, mỗi câu phải mơ hồ theo cách khác nhau.

ĐỊNH DẠNG JSON:
[
  {{
    "messages": [
      {{"role": "system", "content": "Bạn là chuyên gia lịch sử Việt Nam."}},
      {{"role": "user", "content": "QUESTION"}},
      {{"role": "assistant", "content": "ANSWER"}}
    ]
  }}
]

QUAN TRỌNG: CHỈ TRẢ VỀ JSON, KHÔNG THÊM VĂN BẢN NÀO KHÁC.
"""

        return prompt

# Utility functions (giống như file trên)
def get_available_model():
    for model_name in model_priority_list:
        try:
            model = genai.GenerativeModel(model_name)
            print(f"✅ Sử dụng model: {model_name}")
            return model_name
        except Exception as e:
            print(f"❌ Model {model_name} không khả dụng: {e}")
            continue
    return model_priority_list[0]

def clean_json_response(text):
    cleaned = re.sub(r'```json\s*', '', text)
    cleaned = re.sub(r'\s*```', '', cleaned)
    cleaned = cleaned.strip()
    cleaned = re.sub(r'^[^{[]*', '', cleaned)
    cleaned = re.sub(r'[^}\]]*$', '', cleaned)
    return cleaned

def repair_truncated_json(json_str):
    if not json_str.strip():
        return json_str
        
    open_braces = json_str.count('{')
    close_braces = json_str.count('}')
    open_brackets = json_str.count('[')
    close_brackets = json_str.count(']')
    
    repaired = json_str
    
    if open_braces > close_braces:
        repaired += '}' * (open_braces - close_braces)
    if open_brackets > close_brackets:
        repaired += ']' * (open_brackets - close_brackets)
    
    return repaired

def extract_and_parse_json(response_text):
    print(f"🔧 Đang xử lý response dài {len(response_text)} chars...")
    
    cleaned_text = clean_json_response(response_text)
    
    try:
        json_data = json.loads(cleaned_text)
        print("✅ Parse trực tiếp thành công")
        return json_data
    except json.JSONDecodeError:
        print("❌ Parse trực tiếp thất bại")
    
    array_pattern = r'\[\s*\{[\s\S]*?\}\s*\]'
    array_matches = re.findall(array_pattern, cleaned_text, re.DOTALL)
    
    if array_matches:
        json_str = max(array_matches, key=len)
        json_str_repaired = repair_truncated_json(json_str)
        
        try:
            json_data = json.loads(json_str_repaired)
            print("✅ Parse JSON đã sửa thành công")
            return json_data
        except json.JSONDecodeError:
            print("❌ Vẫn lỗi JSON sau khi sửa")
    
    start_idx = cleaned_text.find('[')
    end_idx = cleaned_text.rfind(']')
    
    if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
        json_str = cleaned_text[start_idx:end_idx+1]
        json_str_repaired = repair_truncated_json(json_str)
        
        try:
            json_data = json.loads(json_str_repaired)
            print("✅ Parse JSON manual thành công")
            return json_data
        except json.JSONDecodeError:
            print("❌ Lỗi parse JSON manual")
    
    raise ValueError("Không thể trích xuất JSON từ response")

def generate_clarification_batch(batch_size):
    generator = ClarificationDataGenerator()
    prompt = generator.get_diverse_prompt(batch_size)

    current_model = get_available_model()
    
    for attempt in range(1, max_retry + 1):
        try:
            print(f"🔄 Attempt {attempt} với model {current_model}...")
            
            model = genai.GenerativeModel(current_model)
            response = model.generate_content(
                prompt,
                generation_config=genai.types.GenerationConfig(
                    temperature=temperature,
                    max_output_tokens=max_output_tokens,
                    top_p=0.9
                )
            )
            
            response_text = response.text.strip()
            print(f"📄 Raw response length: {len(response_text)} chars")
            
            json_data = extract_and_parse_json(response_text)
            
            if not isinstance(json_data, list):
                raise ValueError("Kết quả không phải là list")
                
            print(f"✅ Đã tạo được {len(json_data)} samples")
                
            for i, item in enumerate(json_data):
                if "messages" not in item:
                    raise ValueError(f"Thiếu key 'messages' trong item {i}")
                    
            return json_data
            
        except Exception as e:
            print(f"❌ Lỗi attempt {attempt}: {str(e)[:200]}")
            
            if attempt < max_retry:
                next_model_index = (model_priority_list.index(current_model) + 1) % len(model_priority_list)
                current_model = model_priority_list[next_model_index]
                print(f"🔄 Chuyển sang model: {current_model}")
                sleep(3)
            else:
                return []

def create_fallback_clarification_batch(batch_size):
    fallback_data = []
    generator = ClarificationDataGenerator()
    
    question_templates = [
        "Kể về {reference}",
        "{reference} có ý nghĩa gì?",
        "Kết quả của {reference} thế nào?",
        "Ai là {reference}?",
        "{reference} diễn ra khi nào?",
        "Đặc điểm của {reference} là gì?",
        "Ảnh hưởng của {reference} đến lịch sử?",
        "Nguyên nhân của {reference}?"
    ]
    
    for i in range(batch_size):
        # Chọn ngẫu nhiên loại mơ hồ
        vague_type = random.choice(list(generator.vague_references.keys()))
        vague_word = random.choice(generator.vague_references[vague_type])
        
        question_template = random.choice(question_templates)
        question = question_template.format(reference=vague_word)
        
        # Chọn câu trả lời phù hợp với loại mơ hồ
        response_template = random.choice(generator.clarification_responses)
        
        if vague_type == "pronouns":
            example_key = random.choice(["vua", "nhân vật"])
        elif vague_type == "time_periods":
            example_key = "thời kỳ"
        elif vague_type == "locations":
            example_key = "địa điểm" 
        elif vague_type == "events":
            example_key = "sự kiện"
        elif vague_type == "people":
            example_key = random.choice(["vua", "nhân vật"])
        else:
            example_key = "vua"
            
        specific_example = generator.specific_examples[example_key]
        
        if "{specific_example}" in response_template:
            answer = response_template.format(specific_example=specific_example)
        elif "{missing_info}" in response_template:
            answer = response_template.format(missing_info=f"tên, thời gian, hoặc địa điểm cụ thể")
        elif "{aspect}" in response_template:
            answer = response_template.format(aspect=f"khía cạnh cụ thể của {vague_word}")
        else:
            answer = response_template.format(topic=vague_word, context=vague_word, 
                                            detail=vague_word, clarification_point=vague_word,
                                            additional_info=f"thông tin về {vague_word}")
        
        messages = [
            {"role": "system", "content": "Bạn là chuyên gia lịch sử Việt Nam."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer}
        ]
        fallback_data.append({"messages": messages})
    
    return fallback_data

# === XỬ LÝ CHÍNH ===
processed_count = 0
generator = ClarificationDataGenerator()

print(f"🎯 Bắt đầu tạo {target_samples} samples data 'yêu cầu làm rõ' với độ đa dạng cao...")

with open(output_file, "w", encoding="utf-8") as f_out, \
     open(log_file, "w", encoding="utf-8") as f_log:

    while processed_count < target_samples:
        current_batch_size = min(batch_size, target_samples - processed_count)
        print(f"\n🚀 Đang tạo batch {processed_count + 1} đến {processed_count + current_batch_size}...")
        
        batch_data = generate_clarification_batch(current_batch_size)
        
        if not batch_data:
            print("🔄 Sử dụng fallback...")
            batch_data = create_fallback_clarification_batch(current_batch_size)
        
        for data in batch_data:
            f_out.write(json.dumps(data, ensure_ascii=False) + "\n")
            processed_count += 1
        
        f_log.write(f"Đã tạo {processed_count}/{target_samples} samples\n")
        print(f"✅ Đã tạo {processed_count}/{target_samples} samples")
        
        if processed_count < target_samples:
            sleep(5)

print(f"\n🎉 Hoàn tất! Đã tạo {processed_count} samples data 'yêu cầu làm rõ' với độ đa dạng cao.")

🎯 Bắt đầu tạo 3000 samples data 'yêu cầu làm rõ' với độ đa dạng cao...

🚀 Đang tạo batch 1 đến 6...
✅ Sử dụng model: gemini-2.5-flash-lite
🔄 Attempt 1 với model gemini-2.5-flash-lite...
📄 Raw response length: 3349 chars
🔧 Đang xử lý response dài 3349 chars...
✅ Parse trực tiếp thành công
✅ Đã tạo được 6 samples
✅ Đã tạo 6/3000 samples

🚀 Đang tạo batch 7 đến 12...
✅ Sử dụng model: gemini-2.5-flash-lite
🔄 Attempt 1 với model gemini-2.5-flash-lite...
📄 Raw response length: 2763 chars
🔧 Đang xử lý response dài 2763 chars...
✅ Parse trực tiếp thành công
✅ Đã tạo được 6 samples
✅ Đã tạo 12/3000 samples

🚀 Đang tạo batch 13 đến 18...
✅ Sử dụng model: gemini-2.5-flash-lite
🔄 Attempt 1 với model gemini-2.5-flash-lite...
📄 Raw response length: 2990 chars
🔧 Đang xử lý response dài 2990 chars...
✅ Parse trực tiếp thành công
✅ Đã tạo được 6 samples
✅ Đã tạo 18/3000 samples

🚀 Đang tạo batch 19 đến 24...
✅ Sử dụng model: gemini-2.5-flash-lite
🔄 Attempt 1 với model gemini-2.5-flash-lite...
📄 Raw res